# 블로그·카페 통합 전처리 노트북


1. `culumn_name_same`로 블로그 CSV 정규화
2. `blog_cafe` 흐름(병합·형태소 등)
3. `make_stopwords` 흐름(`combined_section_sorted.csv` → PKL)


## 제출용 읽기 안내

**분석 목적**  
블로그와 카페의 컬럼명을 통일하고, 채널·날짜·구간·댓글 구조를 맞춘 뒤 통합 분석용 데이터를 생성한다.

**입력 데이터**  
`data/blog_only/` 블로그 CSV, `data/cafe_only/` 카페 전처리 결과, `data/integrated/combined_section_sorted.csv`

**주요 산출물**  
`data/integrated/crolling_total_estate_press.pkl` 등 통합 분석 입력 파일

**코드 해석 시 유의점**  
이 단계의 핵심은 서로 다른 출처의 데이터를 같은 스키마로 맞추는 것이다.

**해석 작성 칸**  
- 실행 결과에서 확인한 핵심 변화:
- 결과가 연구 질문에 주는 의미:
- 추가로 확인할 점:


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths
PROJECT_ROOT = setup_paths()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "02_integrated"))


# 1 — 컬럼 정규화


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py")

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
from project_paths import DATA_BLOG_ONLY

import importlib.util

_cul = PROJECT_ROOT / "notebooks" / "02_integrated" / "culumn_name_same.py"
spec = importlib.util.spec_from_file_location("culumn_name_same", _cul)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

inp = mod.INPUT_CSV
if inp.is_file():
    mod.save_prepared_csv(inp, mod.OUTPUT_CSV)
    print("OK:", mod.OUTPUT_CSV)
else:
    print("missing input", inp)


# 2 — 블로그·카페 병합


#### 확인용


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd


#### 컬럼명 통일 후 확인


In [ ]:
import sys
import importlib
from pathlib import Path

root = Path.cwd()
while not (root / "project_paths.py").exists() and root != root.parent:
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

_int = root / "notebooks" / "02_integrated"
if str(_int) not in sys.path:
    sys.path.insert(0, str(_int))

from project_paths import DATA_BLOG_ONLY

import culumn_name_same
importlib.reload(culumn_name_same)
from culumn_name_same import prepare_dataframe, resolve_existing_blog_csv

_blog_csv = resolve_existing_blog_csv()
df = prepare_dataframe(_blog_csv)
print("블로그 CSV:", _blog_csv)
display(df.head(10))


In [ ]:
import pandas as pd
from IPython.display import display

from culumn_name_same import prepare_dataframe, resolve_existing_blog_csv

df = prepare_dataframe(resolve_existing_blog_csv())
display(df.head(10))


In [ ]:
comment_df = pd.DataFrame(df.loc[0, "comment_list"])
display(comment_df.head(10))


In [ ]:
comment_table = (
    df[["title", "date", "section", "comment_list"]]
    .explode("comment_list")
    .dropna(subset=["comment_list"])
    .reset_index(drop=True)
)

comment_table["comment_content"] = comment_table["comment_list"].apply(lambda x: x.get("comment_content", ""))
comment_table["comment_like"] = comment_table["comment_list"].apply(lambda x: x.get("comment_like", 0))
comment_table["comment_date"] = comment_table["comment_list"].apply(lambda x: x.get("comment_date", ""))

comment_table = comment_table.drop(columns=["comment_list"])

display(comment_table.head(3))


In [ ]:
comment_df = pd.DataFrame(df.loc[0, "comment_list"])
display(comment_df.head(3))


In [ ]:
import pandas as pd
import json
import ast

from culumn_name_same import resolve_existing_blog_csv

_csv = resolve_existing_blog_csv()
df = pd.read_csv(_csv, encoding="utf-8-sig")
df["comment_list"] = df["comment_list"].apply(json.loads)


In [ ]:
import re
from difflib import SequenceMatcher

# ---------------------------------------------------------
# 1. 텍스트 정제 함수 정의 (조건 1, 2, 4 통합)
# ---------------------------------------------------------
def clean_text(text):
    if pd.isna(text): 
        return ""
    
    text = str(text)
    # 1. HTML 태그 제거
    text = re.sub(r'<[^>]+>', '', text)
    # 2. URL 제거
    text = re.sub(r'http[s]?://\S+', '', text)
    # 3. 자음/모음 반복 (ㅋㅋㅋ, ㅎㅎㅎ, ㅠㅠㅠ 등) 제거
    text = re.sub(r'[ㄱ-ㅎㅏ-ㅣ]+', '', text)
    # 4. 특수문자 및 이모지 제거 (한글, 영문, 숫자, 공백만 남김)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    # 5. 연속된 공백, \n, \t 등을 하나의 공백으로 치환하고 양쪽 여백 제거
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# ---------------------------------------------------------
# 2. comment_list 내부 텍스트 정제 함수
# ---------------------------------------------------------
def clean_comments_list(comments):
    # 1. 만약 리스트가 아니라 문자열('[...]' 형태)로 굳어있다면 리스트로 변환
    if isinstance(comments, str):
        try:
            comments = ast.literal_eval(comments)
        except (ValueError, SyntaxError):
            return comments # 변환 실패 시 원본 유지
            
    # 리스트가 아니면 그대로 반환
    if not isinstance(comments, list):
        return comments
    
    cleaned_comments = []
    for c in comments:
        if not isinstance(c, dict):
            continue
            
        # 2. [핵심 수정] 키 이름 방어 로직
        # 'comment_content'를 먼저 찾고, 없으면 원본 키인 'content'를 찾음
        original_text = c.get('comment_content', c.get('content', ''))
        
        # 텍스트 정제 함수 적용
        cleaned_content = clean_text(original_text)
        
        # 정제 후 내용이 남아있는 경우만 보존
        if cleaned_content:
            new_c = c.copy()
            # 현재 딕셔너리가 갖고 있는 키 이름에 맞춰서 업데이트
            if 'comment_content' in new_c:
                new_c['comment_content'] = cleaned_content
            else:
                new_c['content'] = cleaned_content
                
            cleaned_comments.append(new_c)
            
    return cleaned_comments
# ---------------------------------------------------------
# [실행 파트] 정제 함수 적용
# ---------------------------------------------------------

df['title'] = df['title'].apply(clean_text)
df['doc'] = df['doc'].apply(clean_text)
df['comment_list'] = df['comment_list'].apply(clean_comments_list)

df = df[df['doc'].str.len() >= 5].reset_index(drop=True)


print(f"✅ 1단계 정제 완료!")

# ---------------------------------------------------------
# 3. 고급 중복 제거: 동일 제목 + 본문 90% 이상 일치 (조건 3)
# ---------------------------------------------------------
print("🔍 2단계: 유사도 80% 이상 중복 게시글을 탐색합니다...")

def is_similar(str1, str2, threshold=0.8):
    # SequenceMatcher를 사용해 두 문자열의 유사도 비율(0.0 ~ 1.0) 계산
    return SequenceMatcher(None, str1, str2).ratio() >= threshold

indices_to_drop = set()

# '제목'이 완전히 같은 그룹끼리 묶어서 검사 (속도 최적화)
for title, group in df.groupby('title'):
    if len(group) > 1:
        idxs = group.index.tolist()
        
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                idx1, idx2 = idxs[i], idxs[j]
                
                if idx2 in indices_to_drop:
                    continue
                
                doc1 = df.loc[idx1, 'doc']
                doc2 = df.loc[idx2, 'doc']
                
                if is_similar(doc1, doc2, threshold=0.8):
                    indices_to_drop.add(idx2)

before_cnt = len(df)
df = df.drop(index=list(indices_to_drop)).reset_index(drop=True)
after_cnt = len(df)


print(f"✅ 정제 완료!")
print(f"🗑️ 80% 이상 유사한 중복 게시글 {before_cnt - after_cnt}개 제거됨")
print(f"📊 최종 남은 유효 데이터: {after_cnt}개")


In [ ]:
df.head(20)


In [ ]:
# 변환할 숫자형 컬럼 목록
num_cols = ['like', 'comment_cnt', 'section']

for col in num_cols:
    if col in df.columns:
        # pd.to_numeric을 통해 숫자로 변환 (변환 불가한 값은 NaN 처리 -> 0으로 채움 -> 정수형 변환)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("숫자형 데이터 변환이 완료되었습니다.")
df[num_cols].head()


In [ ]:
import pandas as pd

df = pd.read_csv("naver_blog_medical_quota_preprocessed.csv", encoding="utf-8-sig")
df[["title", "doc", "comment_list"]].head(5)


### 형태소 분석


#### 1. 제목, 본문, 댓글에서 명사,동사,형용사 추출


In [ ]:
import json
import pandas as pd
from konlpy.tag import Okt

# 전처리된 CSV 불러오기
df = pd.read_csv("naver_blog_medical_quota_preprocessed.csv", encoding="utf-8-sig")

# comment_list가 문자열이면 list[dict]로 복원
def parse_comment_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df["comment_list"] = df["comment_list"].apply(parse_comment_list)
df.head()


In [ ]:
%pip install tqdm kiwipiepy konlpy ipywidgets


In [ ]:
import json
import pandas as pd

from tqdm.notebook import tqdm
tqdm.pandas()


In [ ]:
from kiwipiepy import Kiwi
from tqdm.notebook import tqdm

kiwi = Kiwi()


In [ ]:
# Title 결과 저장용
title_token_list = []   # 전체 형태소 분석 결과
title_token_noun = []   # 2글자 이상 명사
title_token_verb = []   # 동사
title_token_adj = []    # 형용사


In [ ]:
for i in tqdm(range(len(df)), desc="title 형태소 분석"):
    text = str(df["title"].iloc[i])

    tokens = kiwi.tokenize(text)

    pos_result = [(t.form, t.tag) for t in tokens]
    title_token_list.append(pos_result)

    nouns = [t.form for t in tokens if t.tag in ["NNG", "NNP"] and len(t.form) > 1]
    title_token_noun.append(nouns)

    verbs = [t.form for t in tokens if t.tag == "VV"]
    title_token_verb.append(verbs)

    adjs = [t.form for t in tokens if t.tag == "VA"]
    title_token_adj.append(adjs)


In [ ]:
# 본문 결과 저장용
doc_token_list = []
doc_token_noun = []
doc_token_verb = []
doc_token_adj = []


In [ ]:
for i in tqdm(range(len(df)), desc="doc 형태소 분석"):
    text = str(df["doc"].iloc[i])

    tokens = kiwi.tokenize(text)

    pos_result = [(t.form, t.tag) for t in tokens]
    doc_token_list.append(pos_result)

    nouns = [t.form for t in tokens if t.tag in ["NNG", "NNP"] and len(t.form) > 1]
    doc_token_noun.append(nouns)

    verbs = [t.form for t in tokens if t.tag == "VV"]
    doc_token_verb.append(verbs)

    adjs = [t.form for t in tokens if t.tag == "VA"]
    doc_token_adj.append(adjs)


In [ ]:
# comment_list가 문자열이면 리스트로 바꾸기
import json
import pandas as pd

def parse_comment_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or not str(value).strip():
        return []
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df["comment_list"] = df["comment_list"].apply(parse_comment_list)


In [ ]:
# 댓글 결과 저장용
comment_text_list = []
comment_token_list = []
comment_token_noun = []
comment_token_verb = []
comment_token_adj = []


In [ ]:
for i in tqdm(range(len(df)), desc="comment 형태소 분석"):
    comments = df["comment_list"].iloc[i]

    row_comment_texts = []
    row_token_list = []
    row_nouns = []
    row_verbs = []
    row_adjs = []

    for item in comments:
        text = str(item.get("comment_content", ""))

        tokens = kiwi.tokenize(text)

        pos_result = [(t.form, t.tag) for t in tokens]
        nouns = [t.form for t in tokens if t.tag in ["NNG", "NNP"] and len(t.form) > 1]
        verbs = [t.form for t in tokens if t.tag == "VV"]
        adjs = [t.form for t in tokens if t.tag == "VA"]

        row_comment_texts.append(text)
        row_token_list.append(pos_result)
        row_nouns.append(nouns)
        row_verbs.append(verbs)
        row_adjs.append(adjs)

    comment_text_list.append(row_comment_texts)
    comment_token_list.append(row_token_list)
    comment_token_noun.append(row_nouns)
    comment_token_verb.append(row_verbs)
    comment_token_adj.append(row_adjs)


In [ ]:
df["title_token_list"] = title_token_list
df["title_token_noun"] = title_token_noun
df["title_token_verb"] = title_token_verb
df["title_token_adj"] = title_token_adj

df["doc_token_list"] = doc_token_list
df["doc_token_noun"] = doc_token_noun
df["doc_token_verb"] = doc_token_verb
df["doc_token_adj"] = doc_token_adj

df["comment_text_list"] = comment_text_list
df["comment_token_list"] = comment_token_list
df["comment_token_noun"] = comment_token_noun
df["comment_token_verb"] = comment_token_verb
df["comment_token_adj"] = comment_token_adj


In [ ]:
df[["doc", "doc_token_list", "doc_token_noun", "doc_token_verb", "doc_token_adj"]].head(5)


In [ ]:
df[["doc", "doc_token_list", "doc_token_noun", "doc_token_verb", "doc_token_adj"]].head(5)


In [ ]:
df[
    [
        "comment_list",
        "comment_text_list",
        "comment_token_list",
        "comment_token_noun",
        "comment_token_verb",
        "comment_token_adj",
    ]
].head(3)


In [ ]:
comment_rows = []

for i in tqdm(range(len(df)), desc="comment 결과 펼치기"):
    texts = df["comment_text_list"].iloc[i]
    token_lists = df["comment_token_list"].iloc[i]
    noun_lists = df["comment_token_noun"].iloc[i]
    verb_lists = df["comment_token_verb"].iloc[i]
    adj_lists = df["comment_token_adj"].iloc[i]

    for j in range(len(texts)):
        comment_rows.append(
            {
                "post_idx": i,
                "comment_idx": j,
                "comment_text": texts[j],
                "comment_token_list": token_lists[j],
                "comment_token_noun": noun_lists[j],
                "comment_token_verb": verb_lists[j],
                "comment_token_adj": adj_lists[j],
            }
        )

comment_df = pd.DataFrame(comment_rows)
comment_df.head(10)


#### 2. 불용어처리


In [ ]:
import os

current_directory = os.getcwd()
print("현재 디렉토리:", current_directory)


In [ ]:
# 불용어 파일 읽기
file_path = os.path.join(current_directory, "stopwords-ko.txt")

f = open(file_path, "r", encoding="utf-8")
st = f.readlines()
f.close()

st[:10]


In [ ]:
# 줄바꿈 제거
stw = []

for i in range(0, len(st)):
    stw.append(st[i].rstrip("\n"))

stw[:20]


In [ ]:
# 검색 속도를 위해 set으로 변환
stw_set = set(stw)

print("기본 불용어 개수:", len(stw_set))


##### 제목


In [ ]:
# 제목 명사에서 기본 불용어 제거
title_token_noun_basic_stw = []

for i in tqdm(range(len(title_token_noun)), desc="제목 기본 불용어 제거"):
    words = title_token_noun[i]
    cleaned_words = [word for word in words if word not in stw_set and len(word) >= 2]
    title_token_noun_basic_stw.append(cleaned_words)

title_token_noun_basic_stw[:3]


##### 본문


In [ ]:
# 본문 명사에서 기본 불용어 제거
doc_token_noun_basic_stw = []

for i in tqdm(range(len(doc_token_noun)), desc="본문 기본 불용어 제거"):
    words = doc_token_noun[i]
    cleaned_words = [word for word in words if word not in stw_set and len(word) >= 2]
    doc_token_noun_basic_stw.append(cleaned_words)

doc_token_noun_basic_stw[:3]


##### 댓글


In [ ]:
# 댓글 명사에서 기본 불용어 제거
comment_token_noun_basic_stw = []

for i in tqdm(range(len(comment_token_noun)), desc="댓글 기본 불용어 제거"):
    row_nouns = comment_token_noun[i]
    row_cleaned = []

    for nouns in row_nouns:
        cleaned_words = [word for word in nouns if word not in stw_set and len(word) >= 2]
        row_cleaned.append(cleaned_words)

    comment_token_noun_basic_stw.append(row_cleaned)

comment_token_noun_basic_stw[:3]


In [ ]:
# df에 저장
df["title_token_noun_basic_stw"] = title_token_noun_basic_stw
df["doc_token_noun_basic_stw"] = doc_token_noun_basic_stw
df["comment_token_noun_basic_stw"] = comment_token_noun_basic_stw


In [ ]:
# 결과 확인
df[["title", "title_token_noun", "title_token_noun_basic_stw"]].head(5)


In [ ]:
df[["doc", "doc_token_noun", "doc_token_noun_basic_stw"]].head(5)


In [ ]:
df[["comment_list", "comment_token_noun", "comment_token_noun_basic_stw"]].head(3)


In [ ]:
# 기본 불용어 제거 결과 저장
df.to_csv("불용어처리_all(blog).csv", index=False, encoding="utf-8-sig")

print("저장 완료: 불용어처리_all(blog).csv")


In [ ]:
noun_only_df = df[["title", "date", "section",
                    "title_token_noun_basic_stw",
                    "doc_token_noun_basic_stw",
                    "comment_token_noun_basic_stw"]].copy()

for col in ["title_token_noun_basic_stw", "doc_token_noun_basic_stw", "comment_token_noun_basic_stw"]:
    noun_only_df[col] = noun_only_df[col].apply(lambda x: json.dumps(x, ensure_ascii=False))

noun_only_df.to_csv("noun_only.csv", index=False, encoding="utf-8-sig")
print("저장 완료: noun_only.csv")


In [ ]:
noun_only_df.tail(10)


#### 블로그랑 카페랑 합치기


In [ ]:
import pandas as pd
from pathlib import Path

base = Path(".")

target_cols = [
    "title",
    "doc",
    "like",
    "comment_cnt",
    "comment_list",
    "ch",
    "date",
    "section",
    "title_token_noun",
    "document_token_noun",
    "comment_token_noun",
]

# 1. CSV 불러오기
blog_df = pd.read_csv(base / "stopwords_removed_all.csv", encoding="utf-8-sig")

# CSV 컬럼명을 피클/최종 스키마에 맞게 통일
blog_df = blog_df.rename(columns={
    "doc_token_noun": "document_token_noun"
})

# 필요한 컬럼만 선택
blog_df = blog_df[target_cols].copy()

# 2. 피클 불러오기
cafe_df = pd.read_pickle(base / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl")

# 필요한 컬럼만 선택
cafe_df = cafe_df[target_cols].copy()

# 3. 세로 결합
combined_df = pd.concat([blog_df, cafe_df], ignore_index=True)

# 4. section 기준 정렬
combined_df["section"] = pd.to_numeric(combined_df["section"], errors="coerce")
combined_df = combined_df.sort_values(by="section", ascending=True).reset_index(drop=True)

# 필요하면 중복 제거
# combined_df = combined_df.drop_duplicates(subset=target_cols).reset_index(drop=True)

# 확인
print(combined_df.head())
print(combined_df.shape)
print(combined_df.columns.tolist())

# 저장
combined_df.to_csv("combined_section_sorted.csv", index=False, encoding="utf-8-sig")
combined_df.to_pickle("combined_section_sorted.pkl")


In [ ]:
combined_df.sample(10)


In [ ]:
import ast
import pandas as pd
from pathlib import Path

base = Path(".")

target_cols = [
    "title",
    "doc",
    "like",
    "comment_cnt",
    "comment_list",
    "ch",
    "date",
    "section",
    "title_token_noun",
    "document_token_noun",
    "comment_token_noun",
]

def parse_maybe_list(x):
    """문자열로 저장된 리스트를 실제 리스트로 변환"""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

def flatten_to_1d(x):
    """이중리스트/혼합리스트를 전부 1차원 리스트로 평탄화"""
    x = parse_maybe_list(x)
    flat = []

    for item in x:
        if isinstance(item, list):
            for sub in item:
                flat.append(sub)
        else:
            flat.append(item)

    return flat

# 1. 블로그 CSV 불러오기
blog_df = pd.read_csv(base / "stopwords_removed_all.csv", encoding="utf-8-sig")

# 컬럼명 통일
blog_df = blog_df.rename(columns={
    "doc_token_noun": "document_token_noun"
})

# 댓글 명사 리스트를 1차원으로 통일
blog_df["comment_token_noun"] = blog_df["comment_token_noun"].apply(flatten_to_1d)

# 제목/본문도 문자열 리스트면 실제 리스트로 통일
blog_df["title_token_noun"] = blog_df["title_token_noun"].apply(parse_maybe_list)
blog_df["document_token_noun"] = blog_df["document_token_noun"].apply(parse_maybe_list)

blog_df = blog_df[target_cols].copy()

# 2. 카페 pkl 불러오기
cafe_df = pd.read_pickle(base / "의대증원_cafedata_total_estate_press_drop_list_pos.pkl")

# 혹시 몰라서 카페도 같은 방식으로 통일
cafe_df["title_token_noun"] = cafe_df["title_token_noun"].apply(parse_maybe_list)
cafe_df["document_token_noun"] = cafe_df["document_token_noun"].apply(parse_maybe_list)
cafe_df["comment_token_noun"] = cafe_df["comment_token_noun"].apply(flatten_to_1d)

cafe_df = cafe_df[target_cols].copy()

# 3. 합치기
combined_df = pd.concat([blog_df, cafe_df], ignore_index=True)

# 4. section 기준 정렬
combined_df["section"] = pd.to_numeric(combined_df["section"], errors="coerce")
combined_df = combined_df.sort_values("section").reset_index(drop=True)

# 확인
print(combined_df[["comment_token_noun"]].head())
print(type(combined_df.loc[0, "comment_token_noun"]))
print(combined_df.shape)

# 저장
combined_df.to_pickle("combined_section_sorted_flat_comments.pkl")
combined_df.to_csv("combined_section_sorted_flat_comments.csv", index=False, encoding="utf-8-sig")


In [ ]:
combined_df.sample(3)


In [ ]:
# 기존 불용어 사전이 없으면 빈 리스트로 시작
if "stw" not in globals():
    stw = []

# 1차: 플랫폼/댓글/기사 메타 잡음
user_stopwords_base = [
    '감사','포스팅','댓글','블로그','방문','답방','이웃','서이추','소통','구경',
    '안녕','오늘','하루','주말','행복','정성','공감','응원','부탁',
    '글','얘기','이야기','말씀','선생','요즘',
    '기사','기자','출처','뉴스','보도','사진','영상','화면','입력','수정',
    '정보','자료','정리','분석','전문','대표',
    '생각','사람','경우','정도','시간','부분','전체','내용','상황','문제',
    '관련','이유','방법','결과','기준','수준','현실','이해','도움','가능','필요','중요',
    '확인','진행','추가','포함','비교','예상','영향','변화','증가','감소',
    '현재','지금','이제','이번','최근','올해','내년','작년','이후',
    '일반','개인','본인','자체','사실','근거','표현','선택','결정','관심','걱정',
    '계획','발표','과정','운영','관리','논의','추진','신청','반영','규모','비율','인원',
    '공개','부분','전체','일부'
]

# 2차: 검색어/주제 앵커 단어
# 검색어 자체가 너무 강해서 주변 담론을 가리는 경우가 많으므로 제거 대상으로 검토
user_stopwords_topic = [
    '의대','증원','의사','의료','정부','대학','정원','병원','환자',
    '교육','정책','교수','전공의','보건','복지부','교육부',
    '대통령','윤석열'
]

# 3차: 입시 편향 완화용
# 현재 데이터는 입시/수험 블로그 비중이 높아서 이 단어들이 계속 반복됩니다.
user_stopwords_admission = [
    '전형','수능','입시','수시','정시','모집','지원','학생','인재',
    '학년도','학교','등급','합격','교과','학원','수학','과목',
    '내신','종합','입학','학과','선발','서울대','학생부',
    '논술','면접','학년','수험','성적','진학','전공','계열',
    '대입','평가','시험','공부','전략','준비'
]

# 4차: 지리/국가/배경어
user_stopwords_geo = [
    '서울','지역','지방','수도','전국',
    '한국','대한민국','국민','국가','나라','사회','세계','미국','일본'
]

# 최대한 많이 거르는 버전
user_stopwords = sorted(set(
    user_stopwords_base +
    user_stopwords_topic +
    user_stopwords_admission +
    user_stopwords_geo
))

# 기존 사전에 합치기
stw = sorted(set(stw + user_stopwords))

print("추가한 사용자 불용어 개수:", len(user_stopwords))
print(user_stopwords[:50])


In [ ]:
import itertools
from collections import Counter

# 이미 만든 사용자 불용어 합치기
user_stopwords = sorted(set(
    user_stopwords_base +
    user_stopwords_topic +
    user_stopwords_admission +
    user_stopwords_geo
))

# 기존 불용어 + 사용자 불용어 합치기
stw = sorted(set(stw + user_stopwords))
base_stopwords = set(stw)

# 명사 원본 만들기: 본문 + 댓글
# 제목까지 넣고 싶으면 title_token_noun도 더하면 됨
df["full_nouns_raw"] = df["document_token_noun"] + df["comment_token_noun"]

# 1차: 기본 불용어 제거
df["full_nouns_base_filtered"] = df["full_nouns_raw"].apply(
    lambda tokens: [w for w in tokens if w not in base_stopwords]
)

# 2차: 구간별 상위 20개 단어를 자동 불용어로 수집
section_top_n = 20
section_top_stopwords = {}

for sec, rows in df.groupby("section")["full_nouns_base_filtered"]:
    tokens = list(itertools.chain.from_iterable(rows))
    top_words = Counter(tokens).most_common(section_top_n)
    section_top_stopwords[sec] = set(word for word, _ in top_words)

# 3차: 전체에서 너무 많이 나오는 단어도 자동 불용어 처리
global_top_n = 30
global_counter = Counter(itertools.chain.from_iterable(df["full_nouns_base_filtered"]))
global_top_stopwords = set(word for word, _ in global_counter.most_common(global_top_n))

# 4차: 최종 불용어 적용
def apply_stopwords(row):
    final_stopwords = (
        base_stopwords
        | global_top_stopwords
        | section_top_stopwords.get(row["section"], set())
    )
    return [w for w in row["full_nouns_raw"] if w not in final_stopwords]

df["full_nouns"] = df.apply(apply_stopwords, axis=1)

# 확인
all_nouns = list(itertools.chain.from_iterable(df["full_nouns"]))
print("최종 남은 명사 예시 50개:")
print(all_nouns[:50])

print("\n전체 고빈도 자동 불용어:")
print(sorted(global_top_stopwords))

for sec, words in section_top_stopwords.items():
    print(f"\n{sec}구간 자동 불용어 20개:")
    print(sorted(words))


In [ ]:
df["full_nouns"] = df.apply(apply_stopwords, axis=1)


### 워드클라우드


In [ ]:
# 데이터 처리용
import pandas as pd
import numpy as np
import itertools

# 문자열 형태의 리스트를 실제 리스트로 바꾸기 위해 사용
from ast import literal_eval

# 단어 빈도 계산용
from collections import Counter

# 주피터에서 표를 보기 좋게 출력하기 위해 사용
from IPython.display import display

# TF-IDF와 코사인 유사도 계산용
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 시각화용
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# 한글이 그래프에서 깨지지 않도록 설정
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
# 앞으로 flat_comments 계열 파일을 쓸 때는 이 파일명만 바꾸면 됩니다.
file_path = "combined_section_sorted_flat_comments.csv"

# CSV 불러오기
df = pd.read_csv(file_path)

# 날짜 컬럼을 날짜형으로 변환
df["date"] = pd.to_datetime(df["date"])

# 데이터 기본 확인
print(f"총 행 수: {len(df):,}")
display(df.head(3))

# section이 실제로 어떤 기간인지 확인
# 현재 데이터 기준:
# 1구간 = 2024-01-01 ~ 2024-03-31
# 2구간 = 2024-04-01 ~ 2024-06-30
# 3구간 = 2024-07-01 ~ 2024-12-31
# 4구간 = 2025-01-01 ~ 2025-06-30
section_range = df.groupby("section")["date"].agg(["min", "max", "count"]).reset_index()
display(section_range)


In [ ]:
import ast
import pandas as pd

def parse_maybe_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []

def flatten_to_1d(x):
    x = parse_maybe_list(x)
    flat = []
    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)
    return flat

df["title_token_noun"] = df["title_token_noun"].apply(parse_maybe_list)
df["document_token_noun"] = df["document_token_noun"].apply(parse_maybe_list)
df["comment_token_noun"] = df["comment_token_noun"].apply(flatten_to_1d)


In [ ]:
for col in ["title_token_noun", "document_token_noun", "comment_token_noun"]:
    print(col, type(df[col].iloc[0]), df[col].iloc[0][:5])


In [ ]:
import sys
import itertools
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation


In [ ]:
# 현재 폴더가 notebooks/ 이든 루트이든 동작하게 프로젝트 루트를 찾습니다.
root = Path.cwd()

while not (root / "project_paths.py").exists() and root != root.parent:
    root = root.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from project_paths import DATA_INTEGRATED, CONFIG_STOPWORDS, OUTPUTS_PIPELINE

OUTPUTS_PIPELINE.mkdir(parents=True, exist_ok=True)


In [ ]:
# 제목 명사까지 포함할지 여부
INCLUDE_TITLE_NOUNS = False

# K-means 클러스터 개수
KMEANS_N_CLUSTERS = 5

# LDA 토픽 개수
LDA_N_TOPICS = 5

# 전체 TF-IDF 상위 몇 개를 볼지
TFIDF_TOP_N = 30

# LDA에서 토픽별 상위 몇 단어를 볼지
LDA_TOP_WORDS = 15

# 워드클라우드 폰트 경로
FONT_PATH = "C:/Windows/Fonts/malgun.ttf"


In [ ]:
# txt 불용어 파일을 읽는 함수
def load_stopwords(path):
    return [
        line.strip()
        for line in path.read_text(encoding="utf-8-sig").splitlines()
        if line.strip()
    ]


In [ ]:
# 워드클라우드를 그리고 필요하면 파일로 저장하는 함수
def make_wordcloud(counter, title, output_file=None):
    wc = WordCloud(
        font_path=FONT_PATH,
        background_color="white",
        width=1400,
        height=900
    ).generate_from_frequencies(dict(counter))

    plt.figure(figsize=(12, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.show()

    if output_file is not None:
        wc.to_file(str(output_file))

    return wc


In [ ]:
# 분석용 원본 명사 컬럼을 만듭니다.
# 기본은 "본문 + 댓글"입니다.
if INCLUDE_TITLE_NOUNS:
    df["full_nouns_raw"] = (
        df["title_token_noun"]
        + df["document_token_noun"]
        + df["comment_token_noun"]
    )
else:
    df["full_nouns_raw"] = (
        df["document_token_noun"]
        + df["comment_token_noun"]
    )


In [ ]:
# 슬라이드 기준:
# "본문 + 댓글"을 하나의 문서 전체 담론으로 사용합니다.
user_stopwords = sorted(set(
    user_stopwords_base +
    user_stopwords_topic +
    user_stopwords_admission +
    user_stopwords_geo
))

stw = sorted(set(stw + user_stopwords))
base_stopwords = set(stw)

df["full_nouns_raw"] = df["document_token_noun"] + df["comment_token_noun"]

# 만약 제목까지 포함하고 싶다면 위 줄 대신 아래 줄을 사용하면 됩니다.
# df["full_nouns_raw"] = df["title_token_noun"] + df["document_token_noun"] + df["comment_token_noun"]

df["full_nouns_base_filtered"] = df["full_nouns_raw"].apply(
    lambda tokens: [w for w in tokens if w not in base_stopwords]
)

section_top_n = 20
section_top_stopwords = {}

for sec, rows in df.groupby("section")["full_nouns_base_filtered"]:
    tokens = list(itertools.chain.from_iterable(rows))
    top_words = Counter(tokens).most_common(section_top_n)
    section_top_stopwords[sec] = set(word for word, _ in top_words)

global_top_n = 30
global_counter = Counter(itertools.chain.from_iterable(df["full_nouns_base_filtered"]))
global_top_stopwords = set(word for word, _ in global_counter.most_common(global_top_n))

def apply_stopwords(row):
    final_stopwords = (
        base_stopwords
        | global_top_stopwords
        | section_top_stopwords.get(row["section"], set())
    )
    return [w for w in row["full_nouns_raw"] if w not in final_stopwords]

df["full_nouns"] = df.apply(apply_stopwords, axis=1)
all_nouns = list(itertools.chain.from_iterable(df["full_nouns"]))

print("최종 남은 명사 예시 50개")
print(all_nouns[:50])


In [ ]:
# 각 sectin(1~4구간)마다 해당 구간의 모든 문서를 합쳐
# 하나의 "구간 문서"로 만드는 단계입니다.
section_df = (
    df.sort_values(["section", "date"])
      .groupby("section")
      .agg(
          period_start=("date", "min"),   # 구간 시작일
          period_end=("date", "max"),     # 구간 종료일
          post_count=("full_nouns", "size"),  # 해당 구간의 문서 수
          all_nouns=("full_nouns", lambda rows: list(itertools.chain.from_iterable(rows)))  # 구간 전체 명사
      )
      .reset_index()
)

# TF-IDF 계산을 위해 명사 리스트를 공백 기준 문자열로 바꿉니다.
section_df["section_doc"] = section_df["all_nouns"].apply(lambda x: " ".join(x))

# 그래프에 쓸 구간 라벨 만들기
section_df["section_label"] = section_df.apply(
    lambda x: f"{int(x['section'])}구간\n({x['period_start'].date()} ~ {x['period_end'].date()})",
    axis=1
)

display(section_df[["section", "period_start", "period_end", "post_count"]])

# 구간별 명사 예시 확인
for _, row in section_df.iterrows():
    print(f"\n===== {int(row['section'])}구간 명사 예시 50개 =====")
    print(row["all_nouns"][:50])


In [ ]:
def top_freq_table(tokens, top_n=20):
    """
    명사 리스트에서 빈도가 높은 단어를 상위 top_n개까지 표로 반환합니다.
    """
    counter = Counter(tokens)
    return pd.DataFrame(counter.most_common(top_n), columns=["keyword", "count"])


# 구간별 상위 빈도 명사 확인
freq_tables = {}

for _, row in section_df.iterrows():
    sec = int(row["section"])
    freq_tables[sec] = top_freq_table(row["all_nouns"], top_n=20)
    
    print(f"\n===== {sec}구간 상위 20개 빈도 명사 =====")
    display(freq_tables[sec])


In [ ]:
# 이미 명사 토큰이 공백으로 이어진 문자열이 있으므로
# tokenizer를 str.split으로 지정해 "띄어쓰기 기준 토큰화"를 합니다.
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

# 4개 구간 문서를 TF-IDF 행렬로 변환
tfidf_matrix = vectorizer.fit_transform(section_df["section_doc"])

# 단어 목록 추출
feature_names = np.array(vectorizer.get_feature_names_out())

print("TF-IDF 행렬 shape:", tfidf_matrix.shape)
print("전체 단어 수:", len(feature_names))


In [ ]:
def top_tfidf_table(matrix, feature_names, row_idx, top_n=20):
    """
    특정 구간(row_idx)의 TF-IDF 값이 높은 단어를 상위 top_n개까지 반환합니다.
    TF-IDF가 높다는 것은:
    - 그 구간에서 자주 나오고
    - 다른 구간과 비교했을 때 상대적으로 특징적인 단어라는 뜻입니다.
    """
    row = matrix[row_idx].toarray().ravel()
    top_idx = row.argsort()[::-1][:top_n]
    
    return pd.DataFrame({
        "keyword": feature_names[top_idx],
        "tfidf": row[top_idx]
    })


# 구간별 TF-IDF 상위 20개 키워드 확인
tfidf_top_tables = {}

for i, sec in enumerate(section_df["section"]):
    sec = int(sec)
    tfidf_top_tables[sec] = top_tfidf_table(tfidf_matrix, feature_names, i, top_n=20)
    
    print(f"\n===== {sec}구간 TF-IDF 상위 20개 키워드 =====")
    display(tfidf_top_tables[sec])


In [ ]:
def hhi_top_n(tokens, top_n=20):
    """
    HHI(Herfindahl-Hirschman Index) 계산 함수
    - 여기서는 '상위 20개 키워드'의 빈도 비중으로 계산합니다.
    - 값이 클수록 소수 키워드에 더 집중되어 있다는 뜻입니다.
    """
    counts = np.array([count for _, count in Counter(tokens).most_common(top_n)], dtype=float)
    
    if counts.sum() == 0:
        return np.nan
    
    shares = counts / counts.sum()
    return np.sum(shares ** 2)


# 구간별 다양성 관련 지표 계산
section_df["noun_count"] = section_df["all_nouns"].apply(len)  # 전체 명사 수
section_df["unique_noun_count"] = section_df["all_nouns"].apply(lambda x: len(set(x)))  # 중복 제거한 명사 수
section_df["hhi_top20"] = section_df["all_nouns"].apply(lambda x: hhi_top_n(x, top_n=20))  # 상위 20개 기준 집중도

metrics_df = section_df[[
    "section", "section_label", "post_count", "noun_count", "unique_noun_count", "hhi_top20"
]].copy()

display(metrics_df)


In [ ]:
# 각 구간의 TF-IDF 상위 20개 키워드를 집합(set)으로 만들어
# 겹친 키워드 / 새로 등장한 키워드 / 사라진 키워드를 비교합니다.
comparison_rows = []

for i in range(len(section_df) - 1):
    a = int(section_df.loc[i, "section"])
    b = int(section_df.loc[i + 1, "section"])
    
    set_a = set(tfidf_top_tables[a]["keyword"])
    set_b = set(tfidf_top_tables[b]["keyword"])
    
    overlap = sorted(set_a & set_b)          # 두 구간에 공통으로 등장한 키워드
    new_words = sorted(set_b - set_a)        # 뒤 구간에서 새로 등장한 키워드
    disappeared = sorted(set_a - set_b)      # 앞 구간에는 있었지만 뒤 구간에서 사라진 키워드
    
    comparison_rows.append({
        "transition": f"{a}→{b}",
        "겹친_키워드_수": len(overlap),
        "새로_등장한_키워드_수": len(new_words),
        "사라진_키워드_수": len(disappeared),
        "겹친_키워드": overlap,
        "새로_등장한_키워드": new_words,
        "사라진_키워드": disappeared
    })

comparison_df = pd.DataFrame(comparison_rows)

# 먼저 개수만 표로 확인
display(comparison_df[["transition", "겹친_키워드_수", "새로_등장한_키워드_수", "사라진_키워드_수"]])

# 실제 키워드 목록도 보기 좋게 출력
for _, row in comparison_df.iterrows():
    print(f"\n===== {row['transition']} 구간 비교 =====")
    print("겹친 키워드:", row["겹친_키워드"])
    print("새로 등장한 키워드:", row["새로_등장한_키워드"])
    print("사라진 키워드:", row["사라진_키워드"])


In [ ]:
def hhi_top_n(tokens, top_n=20):
    """
    상위 20개 키워드의 집중도를 HHI로 계산
    값이 클수록 소수 키워드에 집중된 상태
    """
    counts = np.array([count for _, count in Counter(tokens).most_common(top_n)], dtype=float)
    
    if len(counts) == 0 or counts.sum() == 0:
        return np.nan
    
    shares = counts / counts.sum()
    return np.sum(shares ** 2)

# TF-IDF 벡터 생성
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None
)

tfidf_matrix = vectorizer.fit_transform(section_df["section_doc"])

# 인접 구간 코사인 유사도 계산
similarity_rows = []

for i in range(len(section_df) - 1):
    s1 = int(section_df.loc[i, "section"])
    s2 = int(section_df.loc[i + 1, "section"])
    
    sim = cosine_similarity(tfidf_matrix[i:i+1], tfidf_matrix[i+1:i+2])[0, 0]
    
    similarity_rows.append({
        "transition": f"{s1}→{s2}",
        "cosine_similarity": sim
    })

similarity_df = pd.DataFrame(similarity_rows)

# 시각화에 필요한 지표도 같이 계산
section_df["noun_count"] = section_df["all_nouns"].apply(len)
section_df["unique_noun_count"] = section_df["all_nouns"].apply(lambda x: len(set(x)))
section_df["hhi_top20"] = section_df["all_nouns"].apply(lambda x: hhi_top_n(x, top_n=20))

print("인접 구간 유사도")
print(similarity_df)

print("\n구간별 지표")
section_df[["section", "post_count", "noun_count", "unique_noun_count", "hhi_top20"]]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1) 인접 구간 간 코사인 유사도 라인 차트
axes[0].plot(
    similarity_df["transition"],
    similarity_df["cosine_similarity"],
    marker="o",
    linewidth=2
)
axes[0].set_title("인접 구간 간 코사인 유사도")
axes[0].set_xlabel("구간 전환")
axes[0].set_ylabel("유사도")
axes[0].set_ylim(0, 1)

# 2) 구간별 키워드 집중도(HHI)
axes[1].bar(
    section_df["section_label"],
    section_df["hhi_top20"],
    color="#4C72B0"
)
axes[1].set_title("구간별 키워드 집중도(HHI)")
axes[1].set_xlabel("구간")
axes[1].set_ylabel("HHI")

# 3) 구간별 고유 명사 수
axes[2].bar(
    section_df["section_label"],
    section_df["unique_noun_count"],
    color="#55A868"
)
axes[2].set_title("구간별 고유 명사 수")
axes[2].set_xlabel("구간")
axes[2].set_ylabel("고유 명사 수")

plt.tight_layout()
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1) 인접 구간 간 코사인 유사도 라인 차트
axes[0].plot(
    similarity_df["transition"],
    similarity_df["cosine_similarity"],
    marker="o",
    linewidth=2
)
axes[0].set_title("인접 구간 간 코사인 유사도")
axes[0].set_xlabel("구간 전환")
axes[0].set_ylabel("유사도")
axes[0].set_ylim(0, 1)

# 2) 구간별 키워드 집중도(HHI)
axes[1].bar(
    section_df["section_label"],
    section_df["hhi_top20"],
    color="#4C72B0"
)
axes[1].set_title("구간별 키워드 집중도(HHI)")
axes[1].set_xlabel("구간")
axes[1].set_ylabel("HHI")

# 3) 구간별 고유 명사 수
axes[2].bar(
    section_df["section_label"],
    section_df["unique_noun_count"],
    color="#55A868"
)
axes[2].set_title("구간별 고유 명사 수")
axes[2].set_xlabel("구간")
axes[2].set_ylabel("고유 명사 수")

plt.tight_layout()
plt.show()


In [ ]:
# 워드클라우드용 한글 폰트 경로
font_path = r"C:\Users\이서연\Desktop\2026 1학기\drone_video\nanum-gothic\NanumGothicBold.ttf"

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, (_, row) in zip(axes, section_df.iterrows()):
    # 각 구간의 상위 100개 빈도 명사로 워드클라우드 생성
    freq = dict(Counter(row["all_nouns"]).most_common(100))
    
    wordcloud = WordCloud(
        font_path=font_path,
        background_color="white",
        width=800,
        height=500
    ).generate_from_frequencies(freq)
    
    ax.imshow(wordcloud, interpolation="bilinear")
    ax.set_title(f"{int(row['section'])}구간 ({row['period_start'].date()} ~ {row['period_end'].date()})")
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 이 셀은 matplotlib-venn이 설치되어 있어야 실행됩니다.
# 설치가 안 되어 있으면 아래 줄을 먼저 실행하세요.
%pip install matplotlib-venn

from matplotlib_venn import venn2

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [(1, 2), (2, 3), (3, 4)]

for ax, (a, b) in zip(axes, pairs):
    set_a = set(tfidf_top_tables[a]["keyword"])
    set_b = set(tfidf_top_tables[b]["keyword"])
    
    venn2(
        [set_a, set_b],
        set_labels=(f"{a}구간", f"{b}구간"),
        ax=ax
    )
    ax.set_title(f"TF-IDF 상위 20 키워드 비교: {a}→{b}")

plt.tight_layout()
plt.show()


In [ ]:
# 최종적으로 논문/발표용 해석에 필요한 핵심 표만 다시 모읍니다.
print("1) 인접 구간 코사인 유사도")
display(similarity_df)

print("2) 구간별 다양성/집중도 지표")
display(metrics_df)

print("3) 인접 구간 키워드 변화")
display(comparison_df[["transition", "겹친_키워드_수", "새로_등장한_키워드_수", "사라진_키워드_수"]])


### tf-idf


# 3 — 통합 PKL


## 리스트 수정하기
#### df['ch'] = blog인 행은 comment_token_noun이 이중리스트로 구현돼있음 -> 이중리스트를 하나의 리스트로 통합
#### _token_noun 컬럼들의 값들이 문자열을 값으로 저장하는 리스트로 구현돼있음 -> 리스트 객체로 변환


In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트: notebooks/ 하위 어느 깊이에서 실행해도 동작
_cwd = Path.cwd().resolve()
for _d in [_cwd, *_cwd.parents]:
    if (_d / "notebooks" / "lib" / "notebook_bootstrap.py").is_file():
        sys.path.insert(0, str(_d / "notebooks" / "lib"))
        break
else:
    raise FileNotFoundError("notebooks/lib/notebook_bootstrap.py 을 찾을 수 없습니다. cwd=" + str(_cwd))

from notebook_bootstrap import setup_paths

PROJECT_ROOT = setup_paths()
from project_paths import DATA_CAFE_ONLY, DATA_INTEGRATED, CONFIG_STOPWORDS

# 파일 불러오기
import pandas as pd
import re

df = pd.read_csv(DATA_INTEGRATED / 'combined_section_sorted.csv') 

df.sample(10)


In [ ]:
import ast
import itertools

# 문자열 리스트를 진짜 리스트 객체로 변환하는 함수
def str_to_list(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x

# 모든 대상 컬럼에 변환 함수 적용
df['title_token_noun'] = df['title_token_noun'].apply(str_to_list)
df['document_token_noun'] = df['document_token_noun'].apply(str_to_list)
df['comment_token_noun'] = df['comment_token_noun'].apply(str_to_list)

df.sample(3)


In [ ]:
# 1. 평탄화(Flatten) 함수 정의
def flatten_if_nested(lst):
    # 리스트이고, 그 내부의 첫 번째 요소도 리스트인 경우 (이중 리스트)
    if isinstance(lst, list) and len(lst) > 0 and isinstance(lst[0], list):
        # 중첩된 리스트를 하나로 합침
        return [item for sublist in lst for item in sublist]
    # 이미 평탄화되어 있거나 리스트가 아니면 그대로 반환
    return lst

# 2. 'ch'가 'blog'인 데이터에만 적용
# .loc를 사용해 조건에 맞는 행의 'comment_token_noun' 컬럼만 수정합니다.
mask = df['ch'] == 'blog'
df.loc[mask, 'comment_token_noun'] = df.loc[mask, 'comment_token_noun'].apply(flatten_if_nested)

# 3. 결과 확인
print("✅ blog 데이터 평탄화 완료!")
# blog인 데이터 중 하나를 골라 진짜로 평탄화되었는지(타입이 1차원 리스트인지) 확인
sample_blog = df[df['ch'] == 'blog']['comment_token_noun'].iloc[0]
print(f"샘플 blog 댓글 명사 개수: {len(sample_blog)}개")
print(f"데이터 형태 예시: {sample_blog[:5]}")

df.sample(5)


In [ ]:
# 변환된 파일 저장
import pickle
f = open(DATA_INTEGRATED / "crolling_total_estate_press.pkl", 'wb')
pickle.dump(df, f)
f.close()


In [ ]:
# 파일 불러오기
f = open(DATA_INTEGRATED / "crolling_total_estate_press.pkl", 'rb')
data = pickle.load(f)
f.close()
data


### 추가할 불용어 확인


In [ ]:
# 언패킹(통합) 진행
title_noun = list(itertools.chain(*data['title_token_noun']))
document_noun = list(itertools.chain(*data['document_token_noun']))
comment_noun = list(itertools.chain(*data['comment_token_noun']))

# 결과 확인 
print(title_noun[:10])


In [ ]:
# 상위 빈도 단어 확인
# 제목
from collections import Counter # 단어들을 쉽게 집계하기 위해서 사용
title_count = Counter(title_noun) # 리스트 원소의 개수가 계산됨
title_top = dict(title_count.most_common(100)) # 상위 100개
title_top


In [ ]:
# 본문
document_count = Counter(document_noun) # 리스트 원소의 개수가 계산됨
document_top = dict(document_count.most_common(100)) # 상위 100개
document_top


In [ ]:
# 댓글
comment_count = Counter(comment_noun) # 리스트 원소의 개수가 계산됨
comment_top = dict(comment_count.most_common(100)) # 상위 100개
comment_top
